# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) ??nh v? h? h?i m?t ???ng

??c `README.md` v? `GUIDE.md` tr??c. Notebook ?? tri?n khai E0?E0. Ch?y tr?n Kaggle Notebook v?i GPU T4.


## 0. Môi trường (Kaggle Notebook, GPU T4)


In [ ]:
# Kaggle/Colab: lấy source hỗ trợ từ repo công khai nếu notebook được upload riêng. Bật Internet và chọn GPU trước khi chạy.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ThanhBinh159/K4-Track-4-Day10-NguyenThanhBinh-2A202602777.git"
REPO_NAME = "K4-Track-4-Day10-NguyenThanhBinh-2A202602777"
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "prepare_data.py").is_file():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
    PROJECT_ROOT = WORK_ROOT / REPO_NAME
    if not (PROJECT_ROOT / "scripts" / "prepare_data.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements-colab.txt")], check=True)
print("Project root:", PROJECT_ROOT)


In [ ]:
# Check the libraries used by E0-E4 before inference or training.
from importlib.metadata import PackageNotFoundError, version
import torch
import torchvision
import yaml
from PIL import Image, ImageDraw
from bitsandbytes.nn import Linear4bit
from peft import LoraConfig, get_peft_model
from peft.utils.save_and_load import set_peft_model_state_dict
from safetensors.torch import load_file
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration, get_cosine_schedule_with_warmup

required_versions = {
    "transformers": "5.19.0",
    "peft": "0.21.2",
    "bitsandbytes": "0.50.2",
    "accelerate": "1.15.0",
}
for package, expected in required_versions.items():
    actual = version(package)
    if actual != expected:
        raise RuntimeError(f"{package}={actual}; expected {expected}")
version("kaggle")
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot access the Kaggle GPU")
print("Dependency imports OK; CUDA:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
try:
    torchao_version = version("torchao")
except PackageNotFoundError:
    torchao_version = "not installed"
print("torchao (optional, bypassed for PEFT LoRA):", torchao_version)


In [ ]:
# Đọc Kaggle Secrets trên Kaggle hoặc Colab Secrets trên Colab; không in hay ghi credential vào notebook.
import os
from pathlib import Path

if Path("/kaggle/working").exists():
    from kaggle_secrets import UserSecretsClient
    get_secret = UserSecretsClient().get_secret
else:
    from google.colab import userdata
    get_secret = userdata.get

os.environ["KAGGLE_USERNAME"] = get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = get_secret("KAGGLE_KEY")
del get_secret


## 1. Dữ liệu: tải và chia split

In [ ]:
# Giữ split đã chuẩn bị khi chạy lại notebook trong cùng Kaggle session.
import json
import subprocess
import sys
from pathlib import Path

split_dir = Path("data/splits")
meta_path = split_dir / "meta.json"
ready = meta_path.is_file() and all((split_dir / f"{name}.jsonl").is_file() for name in ("train", "val", "test"))
if ready:
    counts = json.loads(meta_path.read_text(encoding="utf-8"))["counts"]
    ready = sum(1 for _ in Path("data/images").glob("*.jpg")) == sum(item["images"] for item in counts.values())
if ready:
    print("Using existing data/splits and images")
else:
    subprocess.run([sys.executable, "scripts/prepare_data.py", "--kaggle"], check=True)


In [ ]:
# Kiểm tra split, thống kê box và tạo tập val 40 ảnh để chọn setup E4.
import json
import random
import statistics
from pathlib import Path

from scripts.labcommon import CLASSES, read_jsonl, write_jsonl

dataset_meta = json.loads(Path("data/splits/meta.json").read_text(encoding="utf-8"))
split_rows = {
    split: read_jsonl(Path("data/splits") / f"{split}.jsonl")
    for split in ("train", "val", "test")
}
session_sets = {split: {row["session"] for row in rows} for split, rows in split_rows.items()}
assert all(
    not (session_sets[a] & session_sets[b])
    for a, b in (("train", "val"), ("train", "test"), ("val", "test"))
)
print("dataset_version:", dataset_meta["dataset_version"])
box_areas = {label: [] for label in CLASSES}
for split, counts in dataset_meta["counts"].items():
    print(f"{split}: {counts['images']} ảnh; boxes={counts['boxes']}")
for row in split_rows["train"]:
    for obj in row["objects"]:
        box_areas[obj["label"]].append(obj["yolo"][2] * obj["yolo"][3])
median_box_area_pct = {
    label: 100 * statistics.median(areas) if areas else 0.0
    for label, areas in box_areas.items()
}
print("Train median box area (% ảnh):", median_box_area_pct)
stats = {
    "dataset_version": dataset_meta["dataset_version"],
    "counts": dataset_meta["counts"],
    "train_median_box_area_pct": median_box_area_pct,
}
stats_path = Path("report/evidence/dataset_stats.json")
stats_path.parent.mkdir(parents=True, exist_ok=True)
stats_path.write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")

val_rows = split_rows["val"]
val_subset = random.Random(42).sample(val_rows, min(40, len(val_rows)))
val_debug_splits = Path("work/val_subset/splits")
val_debug_splits.mkdir(parents=True, exist_ok=True)
write_jsonl(val_debug_splits / "val.jsonl", val_subset)
(val_debug_splits / "meta.json").write_text(
    json.dumps(dataset_meta, ensure_ascii=False, indent=2), encoding="utf-8"
)
VAL_DEBUG_SPLITS = str(val_debug_splits)
print(f"Validation subset for E4 selection: {len(val_subset)} images")


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Chuyển nhãn YOLO chuẩn hoá sang bbox_2d nguyên trên thang 0–1000.
import json

def yolo_to_box(label, yolo):
    xc, yc, width, height = yolo
    coords = [
        round((xc - width / 2) * 1000),
        round((yc - height / 2) * 1000),
        round((xc + width / 2) * 1000),
        round((yc + height / 2) * 1000),
    ]
    return {"label": label, "bbox_2d": [max(0, min(1000, int(v))) for v in coords]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    boxes.sort(key=lambda item: (item["bbox_2d"][1], item["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

assert yolo_to_box("pothole", [0.5, 0.5, 0.2, 0.4])["bbox_2d"] == [400, 300, 600, 700]
assert objects_to_answer([]) == "[]"


## 3. E0: zero-shot

In [ ]:
# E0: Qwen3-VL zero-shot trên test; ghi raw output để script của đề tự parse/chấm.
import gc
import json
import time
from pathlib import Path

import torch
import yaml
from bitsandbytes.nn import Linear4bit
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

config = yaml.safe_load(Path("configs/e0.yaml").read_text(encoding="utf-8"))
model_id = config["model_id"]
prompt = (
    "Detect every visible road-damage object. Classes: pothole, crack, manhole. "
    "Return only a JSON list; each item must have keys \"label\" and \"bbox_2d\". "
    "bbox_2d is [x1, y1, x2, y2] as integers on a 0-1000 scale relative to image width and height. "
    "Use [] if there are no objects. Do not add any other text."
)
batch_size = 4
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["model.visual", "lm_head"],
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id, dtype=torch.float16, device_map={"": 0}, quantization_config=quantization_config
)
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"
model.eval()

named_params = list(model.named_parameters())
quantized_modules = [(name, module) for name, module in model.named_modules() if isinstance(module, Linear4bit)]
unexpected_quantized = [name for name, _ in quantized_modules if "language_model." not in name]
if unexpected_quantized:
    raise RuntimeError(f"Only LLM layers may be 4-bit; found {unexpected_quantized[:5]}")
packed_correction = sum(module.in_features * module.out_features - module.weight.numel() for _, module in quantized_modules)
visual_params = sum(p.numel() for name, p in named_params if "visual." in name)
projector_params = sum(
    p.numel() for name, p in named_params
    if "visual.merger" in name or "visual.deepstack_merger_list" in name
)
if projector_params < 100_000_000:
    raise RuntimeError(f"Vision projector is still quantized ({projector_params:,} parameters); expected about 109M")
llm_params = sum(
    p.numel() for name, p in named_params
    if "language_model." in name or "lm_head" in name
)
parameter_counts = {
    "vit": visual_params - projector_params,
    "projector": projector_params,
    "llm_embedding_head": llm_params + packed_correction,
    "total": sum(p.numel() for _, p in named_params) + packed_correction,
}
print(
    "Parameter counts:",
    f"ViT={parameter_counts['vit']:,}",
    f"projector={parameter_counts['projector']:,}",
    f"LLM+embedding/head={parameter_counts['llm_embedding_head']:,}",
    f"total={parameter_counts['total']:,}",
)
params_path = Path("report/evidence/model_parameters.json")
params_path.parent.mkdir(parents=True, exist_ok=True)
params_path.write_text(json.dumps(parameter_counts, indent=2), encoding="utf-8")
del named_params, quantized_modules

torch.cuda.reset_peak_memory_stats(0)
entries = [
    json.loads(line)
    for line in Path("data/splits/test.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
pred_path = Path("work/e0.jsonl")
pred_path.parent.mkdir(parents=True, exist_ok=True)
sample_outputs = []
torch.cuda.synchronize()
start_time = time.perf_counter()
with pred_path.open("w", encoding="utf-8") as out_file:
    for start in range(0, len(entries), batch_size):
        batch = entries[start : start + batch_size]
        images = []
        for entry in batch:
            with Image.open(PROJECT_ROOT / "data" / entry["image_path"]) as image:
                images.append(image.convert("RGB"))
        conversations = [
            [{"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ]}]
            for image in images
        ]
        inputs = processor.apply_chat_template(
            conversations, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt", processor_kwargs={"padding": True}
        ).to(torch.device("cuda:0"))
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=config["max_new_tokens"], do_sample=False)
        prompt_length = inputs.input_ids.shape[1]
        raw_outputs = processor.batch_decode(
            generated[:, prompt_length:].cpu(), skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        for entry, raw in zip(batch, raw_outputs):
            row = {"image_id": entry["image_id"], "raw": raw}
            out_file.write(json.dumps(row, ensure_ascii=False) + "\n")
            if len(sample_outputs) < 5:
                sample_outputs.append(row)
        out_file.flush()
        if (start // batch_size + 1) % 10 == 0 or start + len(batch) == len(entries):
            print(f"Generated {start + len(batch)}/{len(entries)} test images")
        del images, conversations, inputs, generated, raw_outputs

torch.cuda.synchronize()
inference_minutes = (time.perf_counter() - start_time) / 60
peak_vram_gb = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
print(f"Saved {len(entries)} predictions to {pred_path}")
print(f"Inference time: {inference_minutes:.2f} min; peak allocated VRAM: {peak_vram_gb:.2f} GB")
print("First five raw outputs:")
for row in sample_outputs:
    print(row["image_id"], row["raw"])
del model, processor
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Chấm E0 đúng thang và tính thêm metric đối chứng khi nhầm output 0–1000 là pixel.
import subprocess
import sys

subprocess.run([
    sys.executable, "scripts/score.py",
    "--pred", "work/e0.jsonl", "--name", "E0",
    "--out", "report/evidence/e0.json",
    "--meta", "vision_precision=fp16",
    "--meta", f"peak_vram_gb={peak_vram_gb:.3f}",
    "--meta", f"inference_minutes={inference_minutes:.3f}",
], check=True)
subprocess.run([
    sys.executable, "scripts/score.py",
    "--pred", "work/e0.jsonl", "--name", "E0-wrong-pixel",
    "--coord", "pixel", "--out", "report/evidence/e0_wrong_coord_pixel.json",
], check=True)
